# Part II — QR Factorization and Least Squares

## Trefethen & Bau, *Numerical Linear Algebra* (1997) — Lecture 6–11

这是《Numerical Linear Algebra》读书笔记的第 2 册。目标不是摘要原书，而是把每一讲整理成一份**可以独立读懂的数值线性代数讲义**，再把它映射到现代 ML systems。

每一讲尽量保持同一结构：数学对象 → 关键公式 → 几何/算法解释 → numerical stability → ML systems mapping → Python experiment。

本册自洽：下面的 setup cell 提供全部依赖，按顺序 run all 即可。

原书 PDF：https://www.stat.uchicago.edu/~lekheng/courses/309/books/Trefethen-Bau.pdf

---

**本系列共 6 册**（Trefethen & Bau, *Numerical Linear Algebra*, 40 Lectures）

| | |
|---|---|
| Part I | [Fundamentals](01_fundamentals.ipynb) |
| Part II | [QR Factorization and Least Squares](02_qr_least_squares.ipynb) |
| Part III | [Conditioning and Stability](03_conditioning_stability.ipynb) |
| Part IV | [Systems of Equations](04_systems_of_equations.ipynb) |
| Part V | [Eigenvalues](05_eigenvalues.ipynb) |
| Part VI | [Iterative Methods](06_iterative_methods.ipynb) |

索引与阅读顺序见 [00_index.ipynb](00_index.ipynb)。


In [1]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import scipy.linalg as sla
    import scipy.sparse.linalg as spla
    SCIPY=True
except Exception:
    SCIPY=False
rng=np.random.default_rng(7)
np.set_printoptions(precision=5,suppress=True)

def relerr(a,b):
    return np.linalg.norm(a-b)/max(np.linalg.norm(b),1e-30)

def stable_rank(A):
    s=np.linalg.svd(A,compute_uv=False)
    return np.sum(s*s)/(s[0]*s[0])

def spectral_norm_power(A,steps=30,seed=0):
    r=np.random.default_rng(seed)
    v=r.normal(size=A.shape[1])
    v/=np.linalg.norm(v)
    for _ in range(steps):
        v=A.T@(A@v)
        v/=np.linalg.norm(v)
    return np.linalg.norm(A@v)

def make_cond(n,kappa,seed=0):
    r=np.random.default_rng(seed)
    Q1,_=np.linalg.qr(r.normal(size=(n,n)))
    Q2,_=np.linalg.qr(r.normal(size=(n,n)))
    s=np.geomspace(1,1/kappa,n)
    return Q1@np.diag(s)@Q2.T

print(f'NumPy {np.__version__} | SciPy {SCIPY}')

NumPy 2.4.2 | SciPy True


## Notation / 贯穿全书的符号

我们主要讨论实矩阵；复数情形把转置 $A^T$ 换成共轭转置 $A^*$。

- $A\in\mathbb{R}^{m\times n}$
- 向量 2-norm：

$$
\Vert x\Vert_2=\sqrt{x^Tx}
$$

- induced matrix 2-norm：

$$
\Vert A\Vert_2=\max_{x\neq0}\frac{\Vert Ax\Vert_2}{\Vert x\Vert_2}=\sigma_{\max}(A)
$$

- Frobenius norm：

$$
\Vert A\Vert_F^2=\sum_{ij}a_{ij}^2=\sum_i\sigma_i^2
$$

- condition number：

$$
\kappa_2(A)=\Vert A\Vert_2\Vert A^{-1}\Vert_2
=\frac{\sigma_{\max}}{\sigma_{\min}}
$$

- unit roundoff：记为 $u$。典型 floating-point model：

$$
\mathrm{fl}(a\circ b)=(a\circ b)(1+\delta),\qquad |\delta|\lesssim u.
$$

计算机算一次加减乘除，不会得到精确的 $a\circ b$，而是得到一个带相对误差的结果。逐项：

- $\mathrm{fl}(\cdot)$：floating-point，机器实际算出来的数；
- $a\circ b$：一次精确运算（$\circ$ 是 $+,-,\times,/$）；
- $\delta$：这次运算引入的相对误差；
- $u$：unit roundoff，这种格式「一次正确舍入」的相对误差上限。

左边是机器结果，右边是「真值再乘 $1+\delta$」。约束的是**相对误差**，不是绝对误差：真值若是 $1.0$，FP32 下次运算大约落在 $1\pm 6\times 10^{-8}$，不会无缘无故错到 $1.01$。

常见 $u$：

- FP32：$u\approx 2^{-24}\approx 6\times 10^{-8}$
- FP16：$u\approx 2^{-11}\approx 5\times 10^{-4}$
- BF16：$u\approx 2^{-8}\approx 4\times 10^{-3}$

一次运算只错 $u$。病态问题可能把这个 $u$ 放大成 $\kappa u$ 量级的解误差。所以不要把「格式很粗」「矩阵很病态」「算法多放大了 rounding」三件事混成一句“数值不稳”。

### 区分

不要把下面三个问题混在一起：

1. **operator amplification**：$\Vert A\Vert$ 大不大？
2. **problem conditioning**：$A^{-1}$ 是否敏感？
3. **algorithm stability**：实现是否额外放大 rounding error？

现代 ML numerics 中，大量争论其实是把这三个层次混在了一起。

# Lecture 6 — Projectors

### 1. Orthogonal projector

若 $Q\in\mathbb{R}^{m\times k}$ 列正交，定义

$$
P=QQ^T.
$$

则

$$
P^2=QQ^TQQ^T=QQ^T=P,
\qquad P^T=P.
$$

对任意 $x$，

$$
x=\underbrace{Px}_{\text{projection}}+\underbrace{(I-P)x}_{\text{residual }r}.
$$

$Px$ 是 $x$ 落在 $\mathcal R(Q)$ 内的分量，**residual $r=(I-P)x$ 就是投影之后剩下的那一部分**。这个命名借自 least squares（见第 2 节）：那里 $r=b-A\hat x=(I-P)b$ 正是标准的残差向量。

它与子空间正交，展开只需要用 $Q^TQ=I_k$：

$$
Q^T(I-P)x=Q^Tx-Q^T\underbrace{QQ^T}_{P}x=Q^Tx-\underbrace{(Q^TQ)}_{I_k}Q^Tx=0.
$$

$Q^Tr=0$ 表示 $r$ 与 $Q$ 每一列都内积为零，而这些列张成 $\mathcal R(Q)$，所以 residual 正交于子空间 $\mathcal R(Q)$，即 $r\in\mathcal R(Q)^\perp$。

顺带，$I-P$ 自己也是正交 projector（投到 $\mathcal R(Q)^\perp$）：$(I-P)^2=I-2P+P^2=I-P$，且 $(I-P)^T=I-P$。于是上面的分解是把空间劈成两个互相正交的部分，勾股定理成立：

$$
\Vert x\Vert_2^2=\Vert Px\Vert_2^2+\Vert (I-P)x\Vert_2^2.
$$

**数值警告**：以上推导全程假设 $Q^TQ=I_k$ 精确成立。实际中 $Q$ 往往是对某个矩阵 $A$ 做 QR 分解算出来的（见 Lecture 7、8），浮点下它并不严格正交，于是 $Q^Tr$ 会偏离 $0$，residual 不再真正垂直于子空间，最小二乘解随之带上误差。偏离多少取决于 $A$ 的条件数 $\kappa_2(A)=\sigma_1/\sigma_n$ 以及所用算法——注意病态的是被分解的 $A$，不是 $Q$ 本身（$Q$ 正交时恒有 $\kappa_2(Q)=1$）。具体的量级对比和数值实验见 **Lecture 8 第 5–7 节**。

### 2. 为什么 projector 是 least squares 的几何核心

寻找 $\mathcal R(A)$ 中离 $b$ 最近的点，就是找 projection：

$$
\hat b=Pb.
$$

如果 $A=QR$，那么 column space 不变：$\mathcal R(A)=\mathcal R(Q)$，所以 projector 直接是 $QQ^T$。

### 3. Representation mapping

比较两个 checkpoint 时，raw basis vector 可能因为任意 rotation 看起来差很大。更稳定的对象往往是 projector：

$$
P_1=Q_1Q_1^T,\qquad P_2=Q_2Q_2^T.
$$

$\Vert P_1-P_2\Vert$ 描述的是 subspace 是否真的改变，而不是坐标系是否旋转。

In [39]:
Q,_=np.linalg.qr(rng.normal(size=(40,6)))
P=Q@Q.T
x=rng.normal(size=40)
a=P@x
b=(np.eye(40)-P)@x
print(f'P^2-P {np.linalg.norm(P @ P - P)} parallel dot perp {a @ b}')

P^2-P 1.2114243084660675e-15 parallel dot perp 7.919057679763142e-16


**ML numerics 自测**

**Q1.** 正交投影 $P=QQ^T$ 满足什么？

**A.** $P^2=P$、$P^T=P$。$x=Px+(I-P)x$，且 residual $(I-P)x$ 正交于 $\mathcal{R}(Q)$。

**Q2.** 它和 least squares 是什么几何关系？

**A.** 在 $\mathcal{R}(A)$ 里找离 $b$ 最近的点就是 $\hat b=Pb$。若 $A=QR$，则 $P=QQ^T$。

**Q3.** 比较两个 checkpoint 的 basis，为什么不该直接比 $Q$？

**A.** $Q$ 可以任意旋转。该比 projector：$\Vert P_1-P_2\Vert$ 说的是 subspace 变没变，不是坐标系转没转。

**Q4.** ML 里什么时候该用 $P$ 而不是 raw activation basis？

**A.** representation drift / subspace stability。把旋转误判成 drift，是常见的假阳性。


# Lecture 7 — QR Factorization

### 1. QR factorization

对 full-column-rank $A\in\mathbb{R}^{m\times n}$（$m\ge n$），

$$
A=QR,
$$

其中

$$
Q^TQ=I,\qquad R\ \text{upper triangular}.
$$

几何上，$Q$ 给出 column space 的 orthonormal basis；$R$ 记录原始 columns 在这个 basis 下的坐标。

“坐标”这句话可以直接验算。取 $A=QR$ 的第 $j$ 列：

$$
a_j=Ae_j=QRe_j=Q\,r_j=\sum_i R_{ij}\,q_i,
$$

其中 $r_j$ 是 $R$ 的第 $j$ 列。这个式子字面就是“$a_j$ 被展开成 $q_i$ 的线性组合，系数为 $R_{ij}$”——而在一组基下的系数就叫坐标。所以 **$R$ 的第 $j$ 列就是 $A$ 的第 $j$ 列在新基 $\{q_i\}$ 下的坐标**。

$R$ 为什么恰好是上三角？因为 $R_{ij}=0\ (i>j)$ 等价于

$$
a_j=\sum_{i\le j}R_{ij}q_i,
$$

即 $a_j$ 只用到前 $j$ 个基向量。等价的说法是**逐层嵌套**：

$$
\mathrm{span}(a_1,\dots,a_j)=\mathrm{span}(q_1,\dots,q_j)\qquad\text{对每个 }j.
$$

这是 QR 区别于 SVD 的地方：SVD 的 $U$ 同样给出 column space 的正交基，却没有“前 $j$ 个恰好张成前 $j$ 列的 span”这种与原始列顺序对齐的结构。上三角不是巧合，而是 Gram–Schmidt 逐列处理这一顺序的直接体现（见 Lecture 8）。

对角元也有几何意义：$|R_{jj}|$ 等于 $a_j$ 中“全新方向”的长度，即 $a_j$ 到 $\mathrm{span}(a_1,\dots,a_{j-1})$ 的距离。$R_{jj}$ 很小说明 $a_j$ 几乎已落在前面各列的 span 内，既是接近秩亏的诊断量，也正是 Lecture 8 中 CGS 发生 catastrophic cancellation 的位置。

### 2. QR 与 least squares

$$
\min_x\Vert Ax-b\Vert_2
=\min_x\Vert QRx-b\Vert_2.
$$

将 $b$ 分解为 $QQ^Tb+(I-QQ^T)b$：

$$
\Vert QRx-b\Vert_2^2
=\Vert Rx-Q^Tb\Vert_2^2+\Vert(I-QQ^T)b\Vert_2^2.
$$

第二项与 $x$ 无关，因此 optimum 满足

$$
Rx=Q^Tb.
$$

这就是 QR least squares。

### 3. Numerical point

QR 本身可以非常 stable，即使 $A$ condition number 很大。要区分：

- factorization 是否 accurate；
- 后续 inverse/least-squares problem 是否 well-conditioned。

**ML numerics 自测**

**Q1.** QR 在几何上分别给出什么？

**A.** $A=QR$：$Q$ 是 column space 的 orthonormal basis，$R$ 是原 columns 在这个 basis 下的坐标。

**Q2.** QR least squares 最后解的是哪个方程？

**A.** $\Vert QRx-b\Vert_2^2=\Vert Rx-Q^Tb\Vert_2^2+\Vert(I-QQ^T)b\Vert_2^2$。第二项与 $x$ 无关，所以 $Rx=Q^Tb$。

**Q3.** QR factorization stable，least squares 就一定准吗？

**A.** 不一定。QR 可以准确地揭示 $A$ 很病态；$R$ 对角很小之后，solve 仍然 sensitive。

**Q4.** 该把“分解准”和“问题好条件”哪件事分开？

**A.** factorization accuracy vs subsequent inverse / least-squares conditioning。两者不是一回事。


# Lecture 8 — Gram-Schmidt Orthogonalization

### 0. 记号约定

本讲从头到尾在做同一件事：**给定输入矩阵 $A$，逐列构造出 Lecture 7 里的 $A=QR$**。三个符号的归属是

$$
A=\big[\,a_1\ a_2\ \cdots\ a_n\,\big]\in\mathbb{R}^{m\times n},
\qquad
Q=\big[\,q_1\ q_2\ \cdots\ q_n\,\big],
\qquad
R=(r_{ij}),
$$

即 **$a_j$ 是待分解矩阵 $A$ 的第 $j$ 列**（输入，已知），$q_i$ 是正在构造的正交基的第 $i$ 列（输出），$r_{ij}$ 是 $R$ 的第 $(i,j)$ 元。算法按 $j=1,2,\dots,n$ 的顺序推进：处理第 $j$ 列时，$q_1,\dots,q_{j-1}$ 已经算好，目标是求出 $q_j$ 和 $R$ 的第 $j$ 列。

由 Lecture 7 第 1 节，$R$ 的第 $j$ 列正是 $a_j$ 在基 $\{q_i\}$ 下的坐标，所以下面每一步算出的 $r_{ij}$ 都是在填 $R$ 的第 $j$ 列。

### 1. Classical Gram–Schmidt

对第 $j$ 列 $a_j$，先算它在已有各基向量上的坐标，减掉这些分量，再把余量归一化：

$$
r_{ij}=q_i^T\,a_j\ \ (i<j),
\qquad
v_j=a_j-\sum_{i<j}r_{ij}q_i,
\qquad
r_{jj}=\Vert v_j\Vert,
\qquad
q_j=\frac{v_j}{r_{jj}}.
$$

注意每个 $r_{ij}$ 的内积里放的都是**原始列 $a_j$**。等价地，这一步是把 Lecture 6 的投影算子直接用一次：

$$
v_j=\big(I-Q_{j-1}Q_{j-1}^T\big)a_j,
\qquad Q_{j-1}=[\,q_1\ \cdots\ q_{j-1}\,].
$$

### 2. Modified Gram–Schmidt

MGS 把上面那一次投影拆成 $j-1$ 次 rank-1 投影，依次从**当前 residual** 中减掉。令 $v_j^{(1)}=a_j$，对 $i=1,\dots,j-1$：

$$
r_{ij}=q_i^T\,v_j^{(i)},
\qquad
v_j^{(i+1)}=v_j^{(i)}-r_{ij}\,q_i,
$$

最后同样取 $r_{jj}=\Vert v_j^{(j)}\Vert$、$q_j=v_j^{(j)}/r_{jj}$。等价地是连续施加一串 rank-1 投影：

$$
v_j=\big(I-q_{j-1}q_{j-1}^T\big)\cdots\big(I-q_1q_1^T\big)a_j.
$$

**两者唯一的区别**：内积里放的是原始列 $a_j$（CGS）还是当前余量 $v_j^{(i)}$（MGS）。

$$
\text{CGS}:\ r_{ij}=q_i^T\,a_j
\qquad\text{vs.}\qquad
\text{MGS}:\ r_{ij}=q_i^T\,v_j^{(i)}
$$

Exact arithmetic 下两者恒等：减掉 $q_1$ 分量后由 $q_2\perp q_1$ 有 $q_2^Tv_j^{(2)}=q_2^Ta_j$，逐项类推。floating point 下算出的 $q_i$ 并非严格正交，这个等式不再成立——MGS 用的 $v_j^{(i)}$ 里含有前面各步已引入的误差，内积能“看见”并部分抵消它；CGS 每次都退回原始 $a_j$，对已犯的误差一无所知，只能任其累积。

> 注意：多数线性代数教材写下的公式（每个投影都作用于原始向量）其实是 CGS，但口头描述“先减掉沿 $q_1$ 的分量、再从剩下的里减掉沿 $q_2$ 的分量……”描述的是 MGS。精确算术下两者相同，所以数学课上不作区分。

### 3. 为什么 CGS 容易失去 orthogonality

当 $a_j$ 已经几乎落在 $\mathrm{span}(q_1,\ldots,q_{j-1})$ 中，

$$
a_j\approx\sum_i r_{ij}q_i.
$$

计算 residual 相当于“大数减大数得到很小的数”，产生 catastrophic cancellation。残余 rounding error 与真正的新方向可能同量级，于是新 $q_j$ 不再正交。

### 3b. cancellation 到底破坏了什么（三层区分）

“rounding 和新方向同量级”容易被理解成“分解失败了”，但实际情况要分三层，结论各不相同。下面的数字来自本讲最后一个 code cell：构造 $60\times4$ 的 $A$，令第 4 列几乎完全落在前 3 列的 span 里（只掺入 $\delta=10^{-14}$ 的新方向），$\kappa(A)\approx8\times10^{14}$。

**第一层：$A=QR$ 仍然成立。**

| | $\Vert A-QR\Vert/\Vert A\Vert$ | $\Vert Q^TQ-I\Vert$ | $R_{44}$ |
|---|---|---|---|
| CGS | 7.0e-17 | **8.0e-02** | 6.6e-14 |
| MGS | 7.3e-17 | **7.6e-02** | 6.6e-14 |
| Householder | 1.9e-16 | 7.4e-16 | 6.7e-14 |

连 CGS 都能精确重建 $A$。坏掉的**只是 $Q$ 的正交性**（$0.08$，已完全不能当正交基用），而 $A=QR$ 这个等式一直满足。失效的是后续所有依赖 $Q^TQ=I$ 的用法：解 $Rx=Q^Tb$、把 $Q$ 当基、算投影 $QQ^T$。这正是 Lecture 7 第 3 节那个区分——**分解准确**与**后续问题好条件**是两回事。

**第二层：$q_j$ 的方向确实不可知，但这与算法无关。** 用最稳定的 Householder，给 $A$ 加相对扰动后看 $q_4$ 偏转多少：

| 扰动幅度 | $q_4$ 偏转角 | $q_1$ 偏转角（对照） |
|---|---|---|
| $10^{-16}$ | 3–5 度 | $10^{-6}$ 度 |
| $10^{-15}$ | 22–26 度 | $10^{-7}$ 度 |
| $10^{-14}$ | 64–85 度 | 0 度 |

机器精度量级的扰动就能让 $q_4$ 转好几度。注意这是 Householder 的结果——**不是算法不行，是 $q_4$ 的方向根本没被数据确定**：$a_4$ 中真正的新成分只有 $\delta=10^{-14}$，在这个精度下不可知。换任何算法都一样。

**第三层：但这不影响分解的可用性。** 因为 $R_{44}\approx6.6\times10^{-14}$ 同样小，$q_4$ 在 $a_4=\sum_i R_{i4}q_i$ 里的权重趋于零，方向再乱也无关紧要——这正是第一层 $\Vert A-QR\Vert\approx10^{-16}$ 的原因。**不可知的部分恰好是不重要的部分。**

所以准确的表述不是“无法分解”，而是：

> 当 $|R_{jj}|$ 降到舍入噪声量级时，$q_j$ 的方向不再由数据决定。$A=QR$ 仍然成立，但 $q_j$ 不应被当作有意义的方向使用——它标志的是**数值秩亏**，即 $A$ 的有效秩小于 $j$。

这同时给出实用判据：**看 $R$ 的对角元**。$|R_{jj}|<\varepsilon\Vert A\Vert$ 即可判定该列数值相关。rank-revealing QR（带列主元，`scipy.linalg.qr(pivoting=True)`）就是把大的 $R_{jj}$ 排到前面，让对角线的衰减直接暴露数值秩。

### 4. ML mapping

- gradient subspace / activation basis；
- Arnoldi basis；
- PCA/low-rank online basis。

如果你看到“新方向”越来越不像新方向，先检查 orthogonality error：

$$
\Vert Q^TQ-I\Vert.
$$


In [46]:
def classical_gram_schmidt(A):
    """A = QR, section 1. Every coefficient r_ij is an inner product with the
    ORIGINAL column a_j, so the projection onto span(q_1..q_{j-1}) is applied in one shot."""
    m,n=A.shape
    Q=np.zeros((m,n),dtype=A.dtype)     # keep A's precision; float64 default would hide fp32 effects
    R=np.zeros((n,n),dtype=A.dtype)
    for j in range(n):
        a_j=A[:,j]
        for i in range(j):
            R[i,j]=Q[:,i]@a_j               # r_ij = q_i^T a_j   <-- original column
        v=a_j-Q[:,:j]@R[:j,j]               # v_j = (I - Q_{j-1} Q_{j-1}^T) a_j
        R[j,j]=np.linalg.norm(v)            # r_jj = length of the genuinely new direction
        Q[:,j]=v/R[j,j]
    return Q,R

def modified_gram_schmidt(A):
    """A = QR, section 2. Identical except each r_ij uses the CURRENT remainder v,
    i.e. a sequence of rank-1 projections instead of one projection onto the span."""
    m,n=A.shape
    Q=np.zeros((m,n),dtype=A.dtype)
    R=np.zeros((n,n),dtype=A.dtype)
    for j in range(n):
        v=A[:,j].copy()
        for i in range(j):
            R[i,j]=Q[:,i]@v                 # r_ij = q_i^T v_j^(i)  <-- current remainder
            v=v-R[i,j]*Q[:,i]               # v_j^(i+1) = v_j^(i) - r_ij q_i
        R[j,j]=np.linalg.norm(v)
        Q[:,j]=v/R[j,j]
    return Q,R

# Vandermonde matrix on a tight interval: columns t^0..t^19 are nearly parallel,
# which makes it severely ill-conditioned -- the classic setting where CGS breaks down.
t=np.linspace(.98,1.02,100)
A=np.vstack([t**j for j in range(20)]).T
I=np.eye(20)
loss=lambda Q: np.linalg.norm(Q.T@Q-I)      # loss of orthogonality
Q_cgs,_=classical_gram_schmidt(A)
Q_mgs,_=modified_gram_schmidt(A)
Q_house,_=np.linalg.qr(A)                   # LAPACK geqrf: Householder reflectors
print(f'cond(A)     {np.linalg.cond(A):.3e}')
print(f'CGS loss         {loss(Q_cgs):.3e}')
print(f'MGS loss        {loss(Q_mgs):.3e}')
print(f'Householder loss {loss(Q_house):.3e}')

cond(A)     1.204e+17
CGS loss         1.553e+01
MGS loss        1.891e+00
Householder loss 2.694e-15


In [43]:
# Sweep kappa_2(A) over 8 orders of magnitude and measure loss of orthogonality ||Q^T Q - I||_2.
# The two Gram-Schmidt variants come from the cell above; this adds one reorthogonalization pass.
def classical_gram_schmidt_twice(A):
    """CGS with a second projection pass -- Kahan's 'twice is enough'."""
    m,n=A.shape
    Q=np.zeros((m,n))
    for j in range(n):
        v=A[:,j]-Q[:,:j]@(Q[:,:j].T@A[:,j])
        v=v-Q[:,:j]@(Q[:,:j].T@v)           # second pass cleans up the first pass's error
        Q[:,j]=v/np.linalg.norm(v)
    return Q

mm,nn=100,30
eps=np.finfo(float).eps
orth=lambda Q: np.linalg.norm(Q.T@Q-np.eye(nn),2)
hdr=f"{'kappa(A)':>10} {'eps*k^2':>10} {'CGS':>10} {'eps*k':>10} {'MGS':>10} {'CGS2':>10} {'House':>10}"
print(hdr)
for p in [2,4,6,8,10]:
    r=np.random.default_rng(0)
    Ua,_=np.linalg.qr(r.normal(size=(mm,nn)))
    Va,_=np.linalg.qr(r.normal(size=(nn,nn)))
    Aq=(Ua*np.logspace(0,-p,nn))@Va.T   # planted spectrum => kappa = 10^p
    kap=np.linalg.cond(Aq)
    Qh,_=np.linalg.qr(Aq)               # LAPACK geqrf uses Householder reflectors
    print(f'{kap:10.2e} {eps*kap**2:10.2e} {orth(classical_gram_schmidt(Aq)[0]):10.2e} '
          f'{eps*kap:10.2e} {orth(modified_gram_schmidt(Aq)[0]):10.2e} '
          f'{orth(classical_gram_schmidt_twice(Aq)):10.2e} {orth(Qh):10.2e}')

  kappa(A)    eps*k^2        CGS      eps*k        MGS       CGS2      House
  1.00e+02   2.22e-12   1.10e-13   2.22e-14   1.04e-14   6.31e-16   1.65e-15
  1.00e+04   2.22e-08   1.65e-09   2.22e-12   6.08e-13   7.00e-16   1.21e-15
  1.00e+06   2.22e-04   1.00e-05   2.22e-10   5.03e-11   6.89e-16   1.10e-15
  1.00e+08   2.22e+00   7.79e-01   2.22e-08   1.58e-08   9.18e-16   1.27e-15
  1.00e+10   2.22e+04   6.42e+00   2.22e-06   6.30e-07   7.10e-16   1.29e-15


### 5. 实测：正交性损失如何随 $\kappa_2(A)$ 增长

上面 cell 的一次运行结果（float64，$m=100$、$n=30$，谱为 $\log$ 均匀分布故 $\kappa=10^p$）：

| $\kappa_2(A)$ | $\varepsilon\kappa^2$ | CGS | $\varepsilon\kappa$ | MGS | CGS2 | Householder |
|---|---|---|---|---|---|---|
| $10^{2}$ | 2.2e-12 | 1.1e-13 | 2.2e-14 | 1.0e-14 | 6.3e-16 | 1.7e-15 |
| $10^{4}$ | 2.2e-08 | 1.7e-09 | 2.2e-12 | 6.1e-13 | 7.0e-16 | 1.2e-15 |
| $10^{6}$ | 2.2e-04 | 1.0e-05 | 2.2e-10 | 5.0e-11 | 6.9e-16 | 1.1e-15 |
| $10^{8}$ | 2.2e+00 | 7.8e-01 | 2.2e-08 | 1.6e-08 | 9.2e-16 | 1.3e-15 |
| $10^{10}$ | 2.2e+04 | 6.4e+00 | 2.2e-06 | 6.3e-07 | 7.1e-16 | 1.3e-15 |

> **这张表的具体数字不可跨环境复现**，换 numpy / BLAS（MKL、OpenBLAS、Accelerate）或线程数都会变。原因是 $U_a,V_a$ 由 `np.linalg.qr` 生成，不同 LAPACK 实现给出不同的正交因子；谱由 `logspace` 硬种进去所以 $\kappa$ 那一列恒定，但矩阵本身不同，舍入行为随之改变。固定谱、只变奇异向量重复 12 次，$\kappa=10^8$ 时 CGS 的损失在 $[2.3\times10^{-2},\ 2.4]$ 之间浮动——**横跨两个数量级**，这种对输入细节的极端敏感本身就是不稳定的表现。同条件下 Householder 的浮动区间不到 2 倍。要看的是各列相对 $\varepsilon\kappa^2$、$\varepsilon\kappa$ 的**量级关系**，不是具体位数。

对应第 1–3 节的理论：

| 算法 | 正交性损失 |
|---|---|
| Classical Gram–Schmidt | $O(\varepsilon\,\kappa_2(A)^2)$ |
| Modified Gram–Schmidt | $O(\varepsilon\,\kappa_2(A))$ |
| CGS + 一次再正交化（CGS2） | $O(\varepsilon)$ |
| Householder QR | $O(\varepsilon)$ |

四点结论：

1. **CGS 实测紧贴 $\varepsilon\kappa^2$**（始终在该上界之下、同数量级内）。$\kappa\ge10^8$ 后损失进入 $O(1)$——这个量级意味着算出的“正交基”根本不正交，Lecture 6 里 $Q^Tr=0$ 那套推导彻底失效。
2. **MGS 紧贴 $\varepsilon\kappa$，好整整一个平方。** 注意 MGS 与 CGS 的浮点运算次数**完全相同**，区别只是第 1、2 节那个内积里放 $a_j$ 还是放当前余量 $v_j^{(i)}$。纯粹的运算顺序重排换来平方级改善。
3. **CGS2 直接达到 $O(\varepsilon)$**，与 Householder 同级，代价是两倍运算量。这就是 Kahan 的 “twice is enough”：投影做两遍就够，做三遍没有额外收益。而且 CGS2 两遍都是矩阵向量乘（BLAS-2），比 MGS 的逐列循环更好向量化，GPU 上往往反而更快。
4. **Householder 在 $\kappa$ 跨 8 个数量级时纹丝不动**，始终 $10^{-15}$。它不是先造基再补救，而是把 $Q$ 表示成一串 reflector 之积，每个都精确正交到 $O(\varepsilon)$，误差无处累积。

### 6. CGS 何时彻底失效：$\kappa_{\text{crit}}\sim\varepsilon^{-1/2}$

令损失 $\varepsilon\kappa^2\sim1$，得临界条件数 $\kappa_{\text{crit}}\sim\varepsilon^{-1/2}$。这个门槛随精度急剧下降：

| 精度 | $\varepsilon$ | $\kappa_{\text{crit}}\sim\varepsilon^{-1/2}$ | 实测崩溃点 |
|---|---|---|---|
| float64 | 2.2e-16 | $6.7\times10^{7}$ | $\kappa=10^8$ 起损失进入 $O(1)$ |
| float32 | 1.2e-07 | $2.9\times10^{3}$ | $\kappa=10^4$ 起损失进入 $O(1)$ |
| bfloat16 | 7.8e-03 | $1.1\times10^{1}$ | —（公式外推） |

> float64 与 float32 两行的实测值由本节代码改 dtype 后测得；bfloat16 一行为公式外推，NumPy 无原生 bf16。

float32 下 $\kappa_2(A)$ 超过约 $10^3$ CGS 就不可用；bfloat16 下临界值低到 $\kappa\approx11$，几乎任何真实矩阵都过不了关。

### 7. 工程结论：实际该用哪个

**默认谁都不用 Gram–Schmidt。** `np.linalg.qr` / `scipy.linalg.qr` 底层是 LAPACK `geqrf`，用的是 Householder reflector 而非 GS。平时写 `Q,R=np.linalg.qr(A)` 拿到的就是表中最后一列的精度。

GS 仍然必要的场合是**列必须一根一根生成**的时候——Krylov 方法（Arnoldi、GMRES、Lanczos）里下一个向量要等上一个正交化完才产生，无法一次性交给 Householder。此时用 MGS 或 CGS2，不要用 CGS。

低精度下尤其要小心：subspace iteration、randomized SVD 的 range finder、在线 PCA、LoRA 初始化的正交化等，在 fp32/bf16 里跑 CGS 基本必然失效。诊断量就是 $\Vert Q^TQ-I\Vert$，出问题时先看它。


In [48]:
# Section 3b: what cancellation actually destroys, when a column is numerically dependent.
# Column 4 lies almost entirely in span(columns 1..3); only delta of a new direction remains.
mr,kr=60,4
rr=np.random.default_rng(3)
Bc=rr.normal(size=(mr,kr-1))
delta=1e-14
Ar=np.column_stack([Bc, Bc@rr.normal(size=kr-1)+delta*rr.normal(size=mr)])
Ir=np.eye(kr)
print(f'kappa(A) = {np.linalg.cond(Ar):.2e}   (column 4 nearly dependent, delta={delta})\n')

# Layer 1: A = QR still holds for every algorithm; only orthogonality of Q differs.
print(f"{'':13s} {'||A-QR||/||A||':>15} {'||Q^TQ-I||':>12} {'R[3,3]':>10}")
for nm,fn in [('CGS',classical_gram_schmidt),('MGS',modified_gram_schmidt),
              ('Householder',np.linalg.qr)]:
    Qr,Rr=fn(Ar)
    print(f'{nm:13s} {np.linalg.norm(Ar-Qr@Rr)/np.linalg.norm(Ar):15.2e} '
          f'{np.linalg.norm(Qr.T@Qr-Ir):12.2e} {abs(Rr[3,3]):10.2e}')

# Layer 2: q_4's direction is not determined by the data -- even with Householder.
print('\nq_4 direction under tiny perturbations of A (Householder):')
Q0,_=np.linalg.qr(Ar)
for er in [1e-16,1e-15,1e-14]:
    ang=[]
    for _ in range(5):
        Ap=Ar+er*np.linalg.norm(Ar)*rr.normal(size=Ar.shape)/np.sqrt(Ar.size)
        Qp,_=np.linalg.qr(Ap)
        ang.append(np.degrees(np.arccos(min(abs(Q0[:,3]@Qp[:,3]),1.0))))
    ref=np.degrees(np.arccos(min(abs(Q0[:,0]@Qp[:,0]),1.0)))
    print(f'  perturbation {er:.0e}: q_4 rotates {np.round(ang,1)} deg   (q_1: {ref:.1e} deg)')

kappa(A) = 7.96e+14   (column 4 nearly dependent, delta=1e-14)

               ||A-QR||/||A||   ||Q^TQ-I||     R[3,3]
CGS                  3.72e-17     1.78e-01   6.67e-14
MGS                  6.21e-17     7.61e-02   6.61e-14
Householder          3.20e-16     7.12e-16   6.65e-14

q_4 direction under tiny perturbations of A (Householder):
  perturbation 1e-16: q_4 rotates [4.6 4.2 3.2 5.3 3.3] deg   (q_1: 1.5e-06 deg)
  perturbation 1e-15: q_4 rotates [25.7 27.2 22.9 27.4 26.3] deg   (q_1: 1.5e-06 deg)
  perturbation 1e-14: q_4 rotates [72.2 79.8 83.  63.7 85.7] deg   (q_1: 8.5e-07 deg)


**ML numerics 自测**

**Q1.** CGS 和 MGS 在 exact arithmetic 下一样吗？

**A.** 一样。floating point 下不一样：MGS 把投影一次次从当前 residual 里减掉，通常更能保住 orthogonality。

**Q2.** CGS 为什么容易丢掉 orthogonality？

**A.** 当 $a_j$ 已几乎在旧 span 里，residual 是大数减大数。cancellation 后，rounding 和真正的新方向同量级。

**Q3.** 最该监控的诊断量是什么？

**A.** $\Vert Q^TQ-I\Vert$。看到“新方向”不像新方向，先查这个。

**Q4.** ML 里哪些地方会踩到同一问题？

**A.** gradient / activation basis、Arnoldi、在线 PCA。病态的是 cancellation，不是“正交这个概念”。


# Lecture 9 — MATLAB → NumPy/PyTorch Numerical Computing

### 1. 原书 MATLAB lecture 在今天对应什么

这里不把重点放在 Python syntax，而是放在**正确调用 numerical kernels**。

优先写：

$$
Ax=b\quad\Rightarrow\quad x=\mathrm{solve}(A,b)
$$

而不是

$$
x=A^{-1}b.
$$

因为显式 inverse：

1. 多做工作；
2. 产生额外 rounding；
3. 丢失 solver 对结构（triangular/SPD）的利用。

### 2. Vectorization 不是只有“快”

`A @ x`, `qr`, `svd`, `lstsq` 背后通常调用成熟 BLAS/LAPACK，这些实现不仅更快，也经过多年数值稳定性设计。

### 3. 编译时代的新问题

PyTorch/Inductor/TensorRT 等 compiler 可能做：

- reassociation；
- fusion；
- precision lowering；
- constant folding；
- kernel substitution。

这些都可能改变 floating-point program。

所以 parity 的正确问题不是“graph mathematically equivalent 吗”，而是：

$$
\text{local numerical perturbation}\times\text{downstream sensitivity}
$$

是否仍在 error budget 内。

In [9]:
A=make_cond(80,1e7,1)
b=rng.normal(size=80)
xs=np.linalg.solve(A,b)
xi=np.linalg.inv(A)@b
print(f'solve residual {relerr(A @ xs, b)}')
print(f'inv@b residual {relerr(A @ xi, b)}')
print(f'solution difference {relerr(xi, xs)}')

solve residual 6.799142404775882e-11
inv@b residual 1.1750344190239224e-10
solution difference 5.618917679728512e-15


**ML numerics 自测**

**Q1.** 为什么不要写 $x=A^{-1}b$，而要 $\mathrm{solve}(A,b)$？

**A.** 显式 inverse 更多工作、更多 rounding，还丢掉 triangular/SPD 等结构。成熟 solver 就是为这个设计的。

**Q2.** vectorized kernel 除了快，还带来什么？

**A.** `@` / `qr` / `svd` / `lstsq` 背后是 BLAS/LAPACK，数值稳定性也经过多年打磨。

**Q3.** compiler 把 graph 变“代数等价”，parity 就该过吗？

**A.** 不该。reassociation、fusion、precision lowering 会改变浮点程序。该问的是 local perturbation $\times$ downstream sensitivity 是否仍在 budget 内。

**Q4.** deployment 里正确的问题是什么？

**A.** 不是 “graph mathematically equivalent 吗”，而是数值扰动传到 task output 后还能否接受。


# Lecture 10 — Householder Triangularization

### 1. Householder reflector

给定非零 $v$，标准形式

$$
H=I-2\frac{vv^T}{v^Tv}.
$$

可验证

$$
H^T=H,
\qquad
H^TH=I,
\qquad
H^2=I.
$$

所以它是一个 orthogonal reflection。

### 2. 如何把一个向量变成 $\pm\Vert x\Vert e_1$

目标是选择 $v$，使

$$
Hx=\alpha e_1,
\qquad |\alpha|=\Vert x\Vert_2.
$$

常取

$$
\alpha=-\operatorname{sign}(x_1)\Vert x\Vert_2,
\qquad
v=x-\alpha e_1.
$$

关键是负号选择：若 $x_1>0$，不去计算 $x_1-\Vert x\Vert$ 这种可能非常小的 cancellation，而使用 $x_1+\Vert x\Vert$。

### 3. QR triangularization

依次作用 Householder：

$$
H_n\cdots H_2H_1A=R.
$$

于是

$$
A=Q R,
\qquad
Q=H_1H_2\cdots H_n.
$$

### 4. Numerical lesson

Householder 的强项不是“数学上能做 QR”，而是可以用一系列 norm-preserving transformations 做到 backward stable QR。

In [10]:
x=rng.normal(size=8)
e1=np.zeros_like(x)
e1[0]=1
alpha=-np.sign(x[0] if x[0]!=0 else 1)*np.linalg.norm(x)
v=x-alpha*e1
v/=np.linalg.norm(v)
H=np.eye(8)-2*np.outer(v,v)
y=H@x
print(f'tail after reflection {np.linalg.norm(y[1:])}')
print(f'orthogonality {np.linalg.norm(H.T @ H - np.eye(8))}')

tail after reflection 4.83984740964111e-16
orthogonality 5.166739843304062e-16


**ML numerics 自测**

**Q1.** Householder $H=I-2vv^T/v^Tv$ 有什么结构？

**A.** $H^T=H$、$H^TH=I$、$H^2=I$：它是正交反射，因此 norm-preserving。

**Q2.** 选 $v$ 时最危险的中间量是什么？

**A.** 若 $x_1>0$，不要算 $x_1-\Vert x\Vert$ 这种可能极小的差。用 $\alpha=-\mathrm{sign}(x_1)\Vert x\Vert_2$，走 $x_1+\Vert x\Vert$，躲开 cancellation。

**Q3.** Householder QR 强在“能做 QR”，还是强在稳定？

**A.** 强在用一串 norm-preserving 变换做出 backward stable QR，而不是仅仅数学上能三角化。

**Q4.** 一串 $H_k$ 为什么不会把前面的误差指数放大？

**A.** 每一步 $\Vert H_k\Vert_2=1$。perturbation 在后续正交变换里不会被越乘越大。


# Lecture 11 — Least Squares Problems

### 1. Least squares 的几何

当 $m>n$ 时，通常不存在 $Ax=b$ 的精确解。我们求

$$
x_*=\arg\min_x\Vert Ax-b\Vert_2.
$$

令 residual

$$
r=b-Ax_*.
$$

最优时 residual 必须与 column space 正交：

$$
A^Tr=0.
$$

因此

$$
A^TAx_*=A^Tb,
$$

即 normal equations。

### 2. 为什么 QR 更自然

若 $A=QR$，

$$
\Vert Ax-b\Vert^2
=\Vert Rx-Q^Tb\Vert^2+\Vert(I-QQ^T)b\Vert^2.
$$

所以直接解

$$
Rx=Q^Tb.
$$

避免显式构造 $A^TA$。

### 3. ML example：linear probe

假设 $A$ 是 frozen embedding matrix，$b$ 是 target。若 embedding columns 高度相关，可能出现：

- prediction $Ax$ 很稳定；
- parameter $x$ checkpoint-to-checkpoint 差异很大。

这不是矛盾，而是 near-null directions 中的 parameter non-identifiability。

In [11]:
A=rng.normal(size=(400,12))
xt=rng.normal(size=12)
b=A@xt+.05*rng.normal(size=400)
x,*_=np.linalg.lstsq(A,b,rcond=None)
r=b-A@x
print(f'||A^T r|| {np.linalg.norm(A.T @ r)} parameter error {relerr(x, xt)}')

||A^T r|| 1.3320492599601637e-12 parameter error 0.0029393932348581293


**ML numerics 自测**

**Q1.** least squares 最优时 residual 满足什么？

**A.** $x_*=\arg\min\Vert Ax-b\Vert_2$。最优则 $A^Tr=0$，即 $r$ 正交于 column space，也就是 normal equations $A^TAx_*=A^Tb$。

**Q2.** 为什么 QR 比显式 $A^TA$ 更自然？

**A.** $A=QR$ 时直接解 $Rx=Q^Tb$，避开构造 $A^TA$（后者把 $\kappa$ 平方）。

**Q3.** linear probe 里 $Ax$ 很稳、$x$ 却乱跳，矛盾吗？

**A.** 不矛盾。near-null directions 上参数不可辨识：预测可以稳，parameter 可以差很多。

**Q4.** 该同时看哪两类量？

**A.** prediction / residual stability，以及 representation / parameter identifiability。只看 loss 会漏掉后者。
